# 01 — Foundations
**Goal:** Understand the two core objects in pandas: `Series` and `DataFrame`.

Everything in pandas is built on these two. Master them and the rest becomes intuitive.

```
Series    → one column of data (1D)
DataFrame → table of data (2D) — a collection of Series sharing the same index
```

In [1]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd

print('pandas version:', pd.__version__)

pandas version: 3.0.3


## 1. Series — a single column

In [2]:
# A Series is a 1D array with an index
# Think of it as a single column from a spreadsheet
sessions = pd.Series([3200, 2800, 3500, 2600, 4100],
                     name='sessions')
print(sessions)
print()
print('dtype:  ', sessions.dtype)   # data type
print('shape:  ', sessions.shape)   # (rows,)
print('index:  ', sessions.index)   # row labels (default: 0,1,2,...)

0    3200
1    2800
2    3500
3    2600
4    4100
Name: sessions, dtype: int64

dtype:   int64
shape:   (5,)
index:   RangeIndex(start=0, stop=5, step=1)


In [3]:
# You can use custom index labels
channels = pd.Series(
    [3.2, 2.1, 4.5, 1.8, 3.8],
    index=['organic', 'paid', 'email', 'social', 'direct'],
    name='cvr_pct'
)
print(channels)
print()
# Access by label
print('Email CVR:', channels['email'])
# Access by position
print('First item:', channels.iloc[0])

organic    3.2
paid       2.1
email      4.5
social     1.8
direct     3.8
Name: cvr_pct, dtype: float64

Email CVR: 4.5
First item: 3.2


In [4]:
# Series operations are vectorized — no loops needed
# All math applies element-wise
print('CVR as decimal:', channels / 100)
print()
print('Above 3%:', channels[channels > 3.0])   # boolean filter
print()
print('Mean:', channels.mean())
print('Max: ', channels.max())
print('Std: ', channels.std())

CVR as decimal: organic    0.032
paid       0.021
email      0.045
social     0.018
direct     0.038
Name: cvr_pct, dtype: float64

Above 3%: organic    3.2
email      4.5
direct     3.8
Name: cvr_pct, dtype: float64

Mean: 3.0800000000000005
Max:  4.5
Std:  1.1344602240713422


## 2. DataFrame — the table

In [5]:
# A DataFrame is a 2D table — rows × columns
# Each column is a Series sharing the same index
df = pd.DataFrame({
    'date':        pd.date_range('2024-01-01', periods=5),
    'channel':     ['organic', 'paid', 'email', 'social', 'direct'],
    'sessions':    [3200, 2800, 3500, 2600, 4100],
    'activations': [96, 59, 158, 47, 156],
})

print(df)
print()
print('Shape:  ', df.shape)           # (rows, columns)
print('Columns:', df.columns.tolist())
print('dtypes:\n', df.dtypes)

        date  channel  sessions  activations
0 2024-01-01  organic      3200           96
1 2024-01-02     paid      2800           59
2 2024-01-03    email      3500          158
3 2024-01-04   social      2600           47
4 2024-01-05   direct      4100          156

Shape:   (5, 4)
Columns: ['date', 'channel', 'sessions', 'activations']
dtypes:
 date           datetime64[us]
channel                   str
sessions                int64
activations             int64
dtype: object


In [6]:
# Quick summary of the data
df.info()     # dtypes + memory usage + null count

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         5 non-null      datetime64[us]
 1   channel      5 non-null      str           
 2   sessions     5 non-null      int64         
 3   activations  5 non-null      int64         
dtypes: datetime64[us](1), int64(2), str(1)
memory usage: 292.0 bytes


In [7]:
# Statistical summary of numeric columns
df.describe()

,date,sessions,activations
count,5,5.000000,5.00000
mean,2024-01-03 00:00:00,3240.000000,103.20000
min,2024-01-01 00:00:00,2600.000000,47.00000
25%,2024-01-02 00:00:00,2800.000000,59.00000
50%,2024-01-03 00:00:00,3200.000000,96.00000
75%,2024-01-04 00:00:00,3500.000000,156.00000
max,2024-01-05 00:00:00,4100.000000,158.00000
std,NaN,594.138031,52.33259


## 3. Accessing columns

In [8]:
# Access a column — returns a Series
print(df['sessions'])           # bracket notation — always works
print()
print(df.sessions)              # dot notation — only works if name has no spaces

# Access multiple columns — returns a DataFrame
print()
print(df[['channel', 'sessions', 'activations']])

0    3200
1    2800
2    3500
3    2600
4    4100
Name: sessions, dtype: int64

0    3200
1    2800
2    3500
3    2600
4    4100
Name: sessions, dtype: int64

   channel  sessions  activations
0  organic      3200           96
1     paid      2800           59
2    email      3500          158
3   social      2600           47
4   direct      4100          156


## 4. Adding and removing columns

In [9]:
# Add a new column — computed from existing ones
df['cvr'] = df['activations'] / df['sessions'] * 100

# Add a constant column
df['country'] = 'CL'

print(df)
print()

# Remove a column
df = df.drop(columns=['country'])
print(df.columns.tolist())

        date  channel  sessions  activations       cvr country
0 2024-01-01  organic      3200           96  3.000000      CL
1 2024-01-02     paid      2800           59  2.107143      CL
2 2024-01-03    email      3500          158  4.514286      CL
3 2024-01-04   social      2600           47  1.807692      CL
4 2024-01-05   direct      4100          156  3.804878      CL

['date', 'channel', 'sessions', 'activations', 'cvr']


## 5. The Index — row labels

In [10]:
# By default the index is 0, 1, 2, ...
print('Default index:', df.index.tolist())

# You can set a column as the index
df_indexed = df.set_index('date')
print()
print(df_indexed)

# Reset back to default integer index
df_reset = df_indexed.reset_index()
print()
print(df_reset.columns.tolist())

Default index: [0, 1, 2, 3, 4]

            channel  sessions  activations       cvr
date                                                
2024-01-01  organic      3200           96  3.000000
2024-01-02     paid      2800           59  2.107143
2024-01-03    email      3500          158  4.514286
2024-01-04   social      2600           47  1.807692
2024-01-05   direct      4100          156  3.804878

['date', 'channel', 'sessions', 'activations', 'cvr']


## 6. Loading real data

In [11]:
# Read the funnel data we already have
df = pd.read_csv('data/raw/funnel_data.csv')

# First look
print('Shape:', df.shape)
print()
df.head()      # first 5 rows

Shape: (450, 12)



,date,channel,visita_landing,inicio_solicitud,datos_personales,otp,datos_financieros,carga_documentos,evaluacion_crediticia,aprobacion,firma_digital,activacion_tarjeta
0,2024-01-01,organic,1010,388,291,207,165,99,89,48,39,27
1,2024-01-01,paid,1021,321,240,163,130,78,70,32,26,18
2,2024-01-01,email,481,202,151,102,89,53,47,25,20,14
3,2024-01-01,social,370,103,77,52,41,22,19,10,8,5
4,2024-01-01,direct,215,86,64,43,34,20,18,10,8,5


In [12]:
df.tail()      # last 5 rows

,date,channel,visita_landing,inicio_solicitud,datos_personales,otp,datos_financieros,carga_documentos,evaluacion_crediticia,aprobacion,firma_digital,activacion_tarjeta
445,2024-03-30,organic,1212,466,349,249,199,119,107,58,47,32
446,2024-03-30,paid,1049,330,247,167,133,79,71,33,27,18
447,2024-03-30,email,492,206,154,104,91,54,48,26,21,14
448,2024-03-30,social,352,98,73,49,39,21,18,9,7,4
449,2024-03-30,direct,213,85,63,42,33,19,17,9,7,4


In [13]:
df.sample(5)   # 5 random rows — good for spotting unexpected values

,date,channel,visita_landing,inicio_solicitud,datos_personales,otp,datos_financieros,carga_documentos,evaluacion_crediticia,aprobacion,firma_digital,activacion_tarjeta
340,2024-03-09,organic,1024,394,295,210,168,100,90,49,40,28
210,2024-02-12,organic,1164,448,336,239,191,114,102,56,45,31
236,2024-02-17,paid,789,248,186,126,100,60,54,25,20,14
40,2024-01-09,organic,947,364,273,194,155,93,83,45,36,25
18,2024-01-04,social,355,99,74,50,40,21,18,9,7,4


In [14]:
# Value counts — how many rows per channel?
print(df['channel'].value_counts())
print()
# Unique values in a column
print('Unique channels:', df['channel'].unique())
print('N unique:       ', df['channel'].nunique())

channel
organic    90
paid       90
email      90
social     90
direct     90
Name: count, dtype: int64

Unique channels: <StringArray>
['organic', 'paid', 'email', 'social', 'direct']
Length: 5, dtype: str
N unique:        5


## Summary
| Concept | Key point |
|---|---|
| `Series` | 1D array with index — one column |
| `DataFrame` | 2D table — collection of Series |
| `df['col']` | Access a column (returns Series) |
| `df[['a','b']]` | Access multiple columns (returns DataFrame) |
| `df['new'] = ...` | Add a column |
| `df.drop(columns=[...])` | Remove columns |
| `df.info()` | dtypes + nulls + memory |
| `df.describe()` | Stats summary of numeric columns |
| `df.head()` / `tail()` / `sample()` | Quick data inspection |
| `df['col'].value_counts()` | Frequency of each value |

**Next:** `02_loading_data.ipynb` — read_csv options, dtypes, Adobe Analytics format.